# 09 — Comparación: original vs VAE/Autoencoder vs f0+loudness

Objetivo del notebook:

1. Cargar un audio de NSynth.
2. Escuchar el audio original.
3. Reconstruirlo con el VAE/autoencoder entrenado, si hay checkpoint disponible.
4. Reconstruir una versión simple usando `f0` + envolvente de energía.
5. Comparar audios y gráficas.

Este notebook está pensado para ejecutarse en otro kernel mientras el entrenamiento largo sigue corriendo en otro notebook.


In [ ]:
# Setup básico
%run init_notebook.py

from pathlib import Path
import sys, os, math

# Asegura que Python encuentra /src aunque ejecutes desde /notebooks
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from src.dataset import NSynth
from src.models import VAE, AutoEncoder
from src.utils.models import compute_magnitude_and_phase, adjust_shape

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Parámetros que puedes tocar

Cambia `IDX` para probar otro audio.

Cambia `MODEL_KIND` a:
- `vae` si quieres cargar un VAE.
- `ae` si quieres cargar un AutoEncoder normal.

El notebook busca checkpoints automáticamente en `data/models`, `models`, `checkpoints` y `outputs`, pero también puedes poner una ruta exacta en `MODEL_PATH`.


In [ ]:
# ---------- PARÁMETROS ----------
IDX = 39
PARTITION = "training"

MODEL_KIND = "vae"     # "vae" o "ae"
MODEL_PATH = None      # ejemplo: "/home/pol/TFG-MUSICAL/data/models/vae.pth"

# Hiperparámetros del modelo STFT que usa 02_VAE.ipynb
SAMPLE_RATE = 16000
N_FFT = 1500
HOP_LENGTH = 250
WIN_LENGTH = N_FFT
INPUT_HEIGHT = 1500
INPUT_WIDTH = 251
INPUT_SIZE = (INPUT_HEIGHT, INPUT_WIDTH)
LATENT_DIM = 200
CHANNELS = [2, 16, 32, 64]

# Parámetros para reconstrucción f0 + loudness
F_MIN = 50
F_MAX = 2000
PERIOD_THRESH = 0.40
CREPE_MODEL = "full"   # "tiny" va más rápido, "full" suele ir mejor


## 2. Cargar muestra del dataset

Esta celda es robusta para las dos versiones del dataset:

- versión antigua: devuelve `waveform, sr, key, metadata`
- versión avanzada: devuelve `waveform, sr, key, metadata, features, condition`


In [ ]:
ds = NSynth(PARTITION)
sample = ds[IDX]

if len(sample) == 4:
    waveform, sr, key, metadata = sample
    features, condition = None, None
elif len(sample) == 6:
    waveform, sr, key, metadata, features, condition = sample
else:
    raise ValueError(f"Formato inesperado del dataset: len(sample)={len(sample)}")

# A mono por seguridad
if waveform.ndim == 2 and waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

waveform = waveform.detach().cpu()
audio = waveform.to(DEVICE)
n_samples = waveform.shape[-1]

print("key:", key)
print("sample rate:", sr)
print("waveform:", waveform.shape)
print("metadata:", metadata)

if condition is not None:
    print("condition keys:", condition.keys())
    print("brightness:", condition["brightness"].item())
    print("sustain:", condition["sustain"].item())

if features is not None:
    print("features keys:", features.keys())
    print("f0 shape:", features["f0"].shape)
    print("loudness shape:", features["loudness_db"].shape)

print("Audio original:")
display(Audio(waveform.squeeze().numpy(), rate=sr))


## 3. Preparar STFT para el VAE/AutoEncoder

El VAE antiguo trabaja con dos canales:

- canal 0: magnitud
- canal 1: fase

Luego reconstruimos la STFT compleja y hacemos inverse STFT para volver a audio.


In [ ]:
stft_transform = T.Spectrogram(
    n_fft=N_FFT,
    win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH,
    power=None,
    onesided=False,
    center=False,
).to(DEVICE)

istft_transform = torchaudio.transforms.InverseSpectrogram(
    n_fft=N_FFT,
    win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH,
    onesided=False,
).to(DEVICE)

with torch.no_grad():
    S = stft_transform(audio)               # (1, 1, F, T), complejo
    mag, phase = compute_magnitude_and_phase(S)
    x = torch.cat([mag, phase], dim=1)      # (1, 2, F, T)

print("STFT:", S.shape)
print("x para el modelo:", x.shape)


## 4. Buscar/cargar checkpoint del modelo

Si no encuentra checkpoint, no pasa nada: saltará la reconstrucción con VAE/AE y podrás seguir con la parte f0+loudness.


In [ ]:
def find_checkpoint():
    if MODEL_PATH is not None and Path(MODEL_PATH).exists():
        return Path(MODEL_PATH)

    candidates = []
    for folder in ["data/models", "models", "checkpoints", "outputs", "examples"]:
        base = PROJECT_ROOT / folder
        if base.exists():
            candidates += list(base.rglob("*.pth"))
            candidates += list(base.rglob("*.pt"))

    if not candidates:
        return None

    # Prioriza nombres con vae/ae según MODEL_KIND
    kind = MODEL_KIND.lower()
    preferred = [p for p in candidates if kind in p.name.lower()]
    return preferred[0] if preferred else candidates[0]

ckpt_path = find_checkpoint()
print("checkpoint encontrado:", ckpt_path)


In [ ]:
model = None
recon_model = None

if ckpt_path is None:
    print("No hay checkpoint. Salto la reconstrucción con modelo.")
else:
    if MODEL_KIND.lower() == "vae":
        model = VAE(input_size=INPUT_SIZE, latent_dim=LATENT_DIM, channels=CHANNELS).to(DEVICE)
    elif MODEL_KIND.lower() == "ae":
        model = AutoEncoder(input_size=INPUT_SIZE, latent_dim=LATENT_DIM, channels=CHANNELS).to(DEVICE)
    else:
        raise ValueError("MODEL_KIND debe ser 'vae' o 'ae'")

    state = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    print("Modelo cargado correctamente.")

    with torch.no_grad():
        if MODEL_KIND.lower() == "vae":
            decoded, kld = model(x)
        else:
            decoded = model(x)

        decoded = adjust_shape(decoded, (INPUT_HEIGHT, INPUT_WIDTH), pad_mode="reflect")

        mag_hat = decoded[:, 0, :, :]
        phase_hat = decoded[:, 1, :, :]

        complex_hat = mag_hat * torch.exp(1j * phase_hat)
        recon_model = istft_transform(complex_hat).detach().cpu()

    # normalización suave para escuchar sin clipping loco
    recon_model = recon_model / (recon_model.abs().max() + 1e-8) * 0.95

    out_path = PROJECT_ROOT / "notebooks" / "recon_model.wav"
    torchaudio.save(str(out_path), recon_model, SAMPLE_RATE)
    print("guardado:", out_path)

    print("Audio reconstruido con", MODEL_KIND)
    display(Audio(recon_model.squeeze().numpy(), rate=SAMPLE_RATE))


## 5. Reconstrucción simple con f0 + loudness/RMS

Esta parte crea una versión muy simple del sonido usando:

- `f0`: frecuencia fundamental estimada con CREPE.
- `RMS`: envolvente de energía aproximada.

No intenta reconstruir el timbre real. Es una senoide controlada por pitch y volumen, así que sonará mucho más pobre que el original. Sirve para comparar qué aporta el modelo frente a una reconstrucción casi solo tonal.


In [ ]:
try:
    import torchcrepe
    HAS_TORCHCREPE = True
except Exception as e:
    HAS_TORCHCREPE = False
    print("No se pudo importar torchcrepe:", repr(e))


In [ ]:
recon_f0 = None

if not HAS_TORCHCREPE:
    print("No hay torchcrepe instalado. Salto reconstrucción f0+loudness.")
else:
    hop_crepe = sr // 100  # 100 fps

    pitch_raw, periodicity = torchcrepe.predict(
        audio,
        sr,
        hop_crepe,
        F_MIN,
        F_MAX,
        model=CREPE_MODEL,
        return_periodicity=True,
        batch_size=512,
        device=DEVICE,
    )

    pitch_raw = pitch_raw.squeeze(0)
    periodicity = periodicity.squeeze(0)

    pitch_raw = torch.nan_to_num(pitch_raw, nan=0.0, posinf=0.0, neginf=0.0)
    periodicity = torch.nan_to_num(periodicity, nan=0.0, posinf=0.0, neginf=0.0)

    # Interpolación de frames poco fiables
    pitch_np = pitch_raw.detach().cpu().numpy().astype(np.float64)
    per_np = periodicity.detach().cpu().numpy()
    pitch_np[per_np < PERIOD_THRESH] = np.nan

    nans = np.isnan(pitch_np)
    if nans.all():
        pitch_np[:] = F_MIN
    elif nans.any():
        idx_arr = np.arange(len(pitch_np))
        pitch_np[nans] = np.interp(idx_arr[nans], idx_arr[~nans], pitch_np[~nans])

    pitch_np = np.clip(pitch_np, F_MIN, F_MAX)
    pitch_filt = torch.tensor(pitch_np, dtype=torch.float64, device=DEVICE)

    # RMS como envolvente de amplitud
    frame_length = 1024
    rms = audio.unfold(-1, frame_length, hop_crepe)
    rms = torch.sqrt(torch.mean(rms ** 2, dim=-1) + 1e-8).squeeze(0).double()
    rms = rms / (rms.max() + 1e-8)

    # Upsampling a número de muestras del audio
    pitch_up = torch.nn.functional.interpolate(
        pitch_filt[None, None, :].float(),
        size=n_samples,
        mode="linear",
        align_corners=False,
    ).squeeze().double()

    rms_up = torch.nn.functional.interpolate(
        rms[None, None, :].float(),
        size=n_samples,
        mode="linear",
        align_corners=False,
    ).squeeze().double()

    phase_acc = 2 * math.pi * torch.cumsum(pitch_up / sr, dim=0)
    sine = 0.25 * rms_up * torch.sin(phase_acc)

    recon_f0 = sine.detach().cpu().float().unsqueeze(0)
    recon_f0 = recon_f0 / (recon_f0.abs().max() + 1e-8) * 0.95

    out_path = PROJECT_ROOT / "notebooks" / "recon_f0_loudness.wav"
    torchaudio.save(str(out_path), recon_f0, sr)
    print("guardado:", out_path)

    print("Audio reconstruido con f0 + loudness/RMS:")
    display(Audio(recon_f0.squeeze().numpy(), rate=sr))


## 6. Gráficas comparativas


In [ ]:
plt.figure(figsize=(12, 3))
plt.plot(waveform.squeeze().numpy()[:4000], label="original", alpha=0.8)

if recon_model is not None:
    plt.plot(recon_model.squeeze().numpy()[:4000], label=f"recon {MODEL_KIND}", alpha=0.7)

if recon_f0 is not None:
    plt.plot(recon_f0.squeeze().numpy()[:4000], label="recon f0+loudness", alpha=0.7)

plt.title("Waveform — zoom primeros 4000 samples")
plt.xlabel("sample")
plt.ylabel("amplitud")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def plot_spec(wav, title, rate):
    spec = torch.stft(
        wav.squeeze(),
        n_fft=1024,
        hop_length=256,
        return_complex=True,
    ).abs()
    spec_db = 20 * torch.log10(spec + 1e-6)
    plt.figure(figsize=(12, 4))
    plt.imshow(spec_db.numpy(), origin="lower", aspect="auto")
    plt.title(title)
    plt.xlabel("frames")
    plt.ylabel("frecuencia")
    plt.colorbar(label="dB")
    plt.show()

plot_spec(waveform, "Espectrograma original", sr)
if recon_model is not None:
    plot_spec(recon_model, f"Espectrograma reconstruido con {MODEL_KIND}", SAMPLE_RATE)
if recon_f0 is not None:
    plot_spec(recon_f0, "Espectrograma reconstruido con f0+loudness", sr)


## 7. Comparación auditiva final


In [ ]:
print("ORIGINAL")
display(Audio(waveform.squeeze().numpy(), rate=sr))

if recon_model is not None:
    print(f"RECONSTRUCCIÓN CON {MODEL_KIND.upper()}")
    display(Audio(recon_model.squeeze().numpy(), rate=SAMPLE_RATE))

if recon_f0 is not None:
    print("RECONSTRUCCIÓN CON F0 + LOUDNESS/RMS")
    display(Audio(recon_f0.squeeze().numpy(), rate=sr))


## Qué deberías apuntar en la memoria

- La reconstrucción con f0+loudness conserva principalmente la altura tonal y la energía temporal, pero pierde casi todo el timbre.
- La reconstrucción con autoencoder/VAE intenta mantener estructura espectral más rica, por eso puede recuperar mejor el carácter del instrumento.
- Si el VAE suena borroso o con artefactos, es esperable: el modelo comprime mucho y reconstruye desde un espacio latente regularizado.
